# TI-501: Jaguar Analysis — Score Distribution

## Overview

**Ticket:** [TI-452](https://mntn.atlassian.net/browse/TI-452)

**Objective:** Compare the distribution of household scores between organic and Jaguar-added IPs in vertical 113002.

**Research Question:** Do Jaguar-added IPs have comparable keyword coverage to organic IPs?

**Approach:**
1. Filter `ip_vertical_associations` to vertical 113002
2. Join to `prospecting_intent` to get household scores
3. Categorize scores into 4 biddable groups
4. Compare distribution between `is_model_added` = TRUE (Jaguar) vs FALSE (Organic)

## Score Categories

| Score | Name | Meaning | Biddable? |
|-------|------|---------|----------|
| 10000 | HI (High Intent) | Vertical + Keywords | Yes |
| 8000 | Peak Performance | Vertical only (no keywords) | Yes |
| 3333-6665 | MI (Mid Intent) | Bucket + Keywords (not vertical) | Yes (if has keywords) |
| Other | Max Reach | Keywords only (outside bucket/vertical) | Yes (if has keywords) |

**Hypothesis:** If Jaguar-added IPs skew toward Peak Performance (8000) vs HI (10000), they have fewer keyword matches than organic IPs.

## Performance Note
The `prospecting_intent` table is ~242 billion rows. This notebook avoids caching large joins to prevent disk space issues.

---
## 1. Setup & Configuration

In [0]:
# =============================================================================
# IMPORTS
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum as spark_sum,
    when,
    round as spark_round,
)
import pandas as pd

# Initialize Spark session
spark = SparkSession.builder.appName("jaguar_score_analysis").getOrCreate()

# Disable disk caching to avoid space issues with large tables
spark.conf.set("spark.databricks.io.cache.enabled", "false")

print("Setup complete.")

Setup complete.


In [0]:
# =============================================================================
# CONFIGURATION PARAMETERS
# =============================================================================

# Analysis date (single date for score snapshot)
TARGET_DATE = "2025-11-25"

# Vertical filter
# - Specific vertical: [113002]
# - Multiple verticals: [113002, 113003]
# - All verticals: None
VERTICAL_IDS = [113002]

# =============================================================================
# DERIVED PATHS
# =============================================================================

IP_VERT_PATH = f"gs://mntn-data-archive-prod/vertical_categorizations/ip_vertical_associations/dt={TARGET_DATE}/"

# Parse date for prospecting path (handles zero-padding)
year = TARGET_DATE[:4]
month = TARGET_DATE[5:7]
day = TARGET_DATE[8:10]
PROSPECTING_PATH = f"gs://household-scoring-prod/output/scoring/prospecting_intent/year={year}/month={month}/day={day}/"

print(f"Target Date:      {TARGET_DATE}")
print(f"Vertical IDs:     {VERTICAL_IDS if VERTICAL_IDS else 'ALL'}")
print(f"IP Vert Path:     {IP_VERT_PATH}")
print(f"Prospecting Path: {PROSPECTING_PATH}")

Target Date:      2025-11-25
Vertical IDs:     [113002]
IP Vert Path:     gs://mntn-data-archive-prod/vertical_categorizations/ip_vertical_associations/dt=2025-11-25/
Prospecting Path: gs://household-scoring-prod/output/scoring/prospecting_intent/year=2025/month=11/day=25/


---
## 2. Load IP Vertical Associations

Filter to target vertical(s) and extract distinct IPs with their `is_model_added` flag.

In [0]:
# Load IP vertical associations
ip_vert_raw_df = spark.read.parquet(IP_VERT_PATH)

# Filter to target vertical(s)
if VERTICAL_IDS:
    ip_vert_df = ip_vert_raw_df.filter(
        col("data_source_category_id").isin([float(v) for v in VERTICAL_IDS])
    )
else:
    ip_vert_df = ip_vert_raw_df

# Get distinct IP + is_model_added combinations
# Cache this smaller table (should be ~23M rows)
vertical_ips_df = (
    ip_vert_df
    .select("ip", "is_model_added")
    .distinct()
    .cache()
)

# Validation
total_ips = vertical_ips_df.count()
print(f"Total IPs in vertical: {total_ips:,}")

Total IPs in vertical: 22,823,104


In [0]:
# Validation: IP distribution by is_model_added
print("IP Distribution by is_model_added:")
print("=" * 50)

(
    vertical_ips_df
    .groupBy("is_model_added")
    .agg(count("*").alias("ip_count"))
    .orderBy("is_model_added")
    .show()
)

IP Distribution by is_model_added:
+--------------+--------+
|is_model_added|ip_count|
+--------------+--------+
|         false|22337208|
|          true|  485896|
+--------------+--------+



---
## 3. Load Prospecting Intent & Calculate Score Distribution

**Important:** We do NOT cache the joined result to avoid disk space issues.
Instead, we perform the aggregation in a single pass.

In [0]:
# Load prospecting intent scores
prospecting_df = spark.read.parquet(PROSPECTING_PATH)

print("Prospecting intent schema:")
prospecting_df.printSchema()

Prospecting intent schema:
root
 |-- ip: string (nullable = true)
 |-- advertiser_id: integer (nullable = true)
 |-- campaign_group_id: integer (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- household_score: integer (nullable = true)



In [0]:
# Join and aggregate in a single pass - NO CACHING of the large result
# This avoids disk space issues with the 242B row table

# Step 1: Inner join to filter prospecting_intent to only our IPs
# Step 2: Categorize scores
# Step 3: Aggregate - all in one query

score_distribution_df = (
    prospecting_df
    .join(vertical_ips_df, on="ip", how="inner")
    .withColumn(
        "score_category",
        when(col("household_score") == 10000, "HI")
        .when(col("household_score") == 8000, "Peak Performance")
        .when(col("household_score").between(3333, 6665), "MI")
        .otherwise("Max Reach")
    )
    .groupBy("is_model_added", "score_category")
    .agg(count("*").alias("score_count"))
)

# Collect the aggregated results (this is small - only 8 rows max)
score_counts_collected = score_distribution_df.collect()

print(f"Aggregation complete. Retrieved {len(score_counts_collected)} group counts.")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-4767014528524285>, line 23
      8 score_distribution_df = (
      9     prospecting_df
     10     .join(vertical_ips_df, on="ip", how="inner")
   (...)
     19     .agg(count("*").alias("score_count"))
     20 )
     22 # Collect the aggregated results (this is small - only 8 rows max)
---> 23 score_counts_collected = score_distribution_df.collect()
     25 print(f"Aggregation complete. Retrieved {len(score_counts_collected)} group counts.")

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databr

---
## 4. Calculate Percentages & Format Results

In [0]:
# Convert collected results to pandas for easier manipulation
score_dist_pd = pd.DataFrame(
    [(row["is_model_added"], row["score_category"], row["score_count"]) 
     for row in score_counts_collected],
    columns=["is_model_added", "score_category", "score_count"]
)

# Calculate totals per is_model_added group
totals = score_dist_pd.groupby("is_model_added")["score_count"].sum().reset_index()
totals.columns = ["is_model_added", "total"]

# Merge totals and calculate percentage
score_dist_pd = score_dist_pd.merge(totals, on="is_model_added")
score_dist_pd["pct"] = (score_dist_pd["score_count"] / score_dist_pd["total"] * 100).round(2)

# Map boolean to readable names
score_dist_pd["group"] = score_dist_pd["is_model_added"].map(
    {True: "Model-Added", False: "Organic"}
)

print("Score distribution calculated.")
display(score_dist_pd)

In [0]:
# Pivot: Score categories as rows, groups as columns
pivot_counts = score_dist_pd.pivot(
    index="score_category",
    columns="group",
    values="score_count"
).fillna(0).astype(int)

pivot_pct = score_dist_pd.pivot(
    index="score_category",
    columns="group",
    values="pct"
).fillna(0)

# Calculate percent difference
if "Organic" in pivot_pct.columns and "Model-Added" in pivot_pct.columns:
    pivot_pct["Pct_Diff"] = (
        (pivot_pct["Model-Added"] - pivot_pct["Organic"]) 
        / pivot_pct["Organic"].replace(0, float('nan')) 
        * 100
    ).round(2)

# Reorder score categories logically
category_order = ["HI", "Peak Performance", "MI", "Max Reach"]
pivot_counts = pivot_counts.reindex(category_order)
pivot_pct = pivot_pct.reindex(category_order)

---
## 5. Display Results

In [0]:
# Display results
print("=" * 70)
print("JAGUAR SCORE ANALYSIS: Model-Added vs Organic")
print(f"Date: {TARGET_DATE} | Vertical(s): {VERTICAL_IDS}")
print("=" * 70)

print("\n📊 RAW COUNTS (IP × Campaign combinations)")
print("-" * 50)
display(pivot_counts)

print("\n📈 PERCENTAGE DISTRIBUTION")
print("-" * 50)
display(pivot_pct)

In [0]:
# Summary statistics
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

organic_hi = pivot_pct.loc["HI", "Organic"] if "Organic" in pivot_pct.columns else 0
model_hi = pivot_pct.loc["HI", "Model-Added"] if "Model-Added" in pivot_pct.columns else 0
organic_pp = pivot_pct.loc["Peak Performance", "Organic"] if "Organic" in pivot_pct.columns else 0
model_pp = pivot_pct.loc["Peak Performance", "Model-Added"] if "Model-Added" in pivot_pct.columns else 0

print(f"\nHI Rate (Vertical + Keywords):")
print(f"  Organic:      {organic_hi:.1f}%")
print(f"  Model-Added:  {model_hi:.1f}%")
print(f"  Difference:   {model_hi - organic_hi:+.1f} ppts")

print(f"\nPeak Performance Rate (Vertical only):")
print(f"  Organic:      {organic_pp:.1f}%")
print(f"  Model-Added:  {model_pp:.1f}%")
print(f"  Difference:   {model_pp - organic_pp:+.1f} ppts")

# Interpretation
print("\n" + "-" * 70)
print("INTERPRETATION")
print("-" * 70)

if model_hi > organic_hi:
    print("✅ Jaguar-added IPs have a HIGHER HI rate than organic IPs.")
    print("   This suggests Jaguar is selecting IPs with strong keyword coverage.")
elif model_hi < organic_hi:
    print("⚠️  Jaguar-added IPs have a LOWER HI rate than organic IPs.")
    print("   This suggests Jaguar-added IPs have weaker keyword coverage.")
else:
    print("➡️  HI rates are comparable between groups.")

---
## 6. Export Results

In [0]:
# Prepare export DataFrame
export_df = pivot_pct.reset_index().rename(columns={"score_category": "Score Category"})

print("Export-ready format:")
display(export_df)

# Uncomment to save to CSV
# export_path = f"/dbfs/tmp/jaguar_score_analysis_{TARGET_DATE}.csv"
# export_df.to_csv(export_path, index=False)
# print(f"\nSaved to: {export_path}")

---
## 7. Cleanup

In [0]:
# # Uncache DataFrames to free memory
# vertical_ips_df.unpersist()

# # Re-enable disk caching for other workloads
# spark.conf.set("spark.databricks.io.cache.enabled", "true")

# print("Cleanup complete.")